In [ ]:
from pathlib import Path
import argparse
import os
import time

import numpy as np
from pynq import Overlay, allocate


MAX_HEIGHT = 128
MAX_WIDTH = 256
ALPHA_VAL = 1.0
N_ITER = 20
FLOW_FRAC_BITS = 10
OUT_FLOW_ARROWS_NAME = "pynq_out_flow_arrows.png"
FLOW_ARROWS_STEP = 5
FLOW_ARROWS_SCALE = 5.0
FLOW_ARROWS_THICKNESS = 1
FLOW_ARROWS_TIP_LENGTH = 0.3

REG_CTRL = 0x00
REG_AP_RETURN = 0x10
REG_INP1_IMG_LO = 0x18
REG_INP1_IMG_HI = 0x1C
REG_INP2_IMG_LO = 0x24
REG_INP2_IMG_HI = 0x28
REG_VX_IMG_LO = 0x30
REG_VX_IMG_HI = 0x34
REG_VY_IMG_LO = 0x3C
REG_VY_IMG_HI = 0x40
REG_HEIGHT = 0x48
REG_WIDTH = 0x50


def _find_hls_ip(overlay: Overlay):
    for name, info in overlay.ip_dict.items():
        ip_type = info.get("type", "")
        if "xilinx.com:hls:hls_HS" in ip_type:
            return name
    for name in overlay.ip_dict.keys():
        if "hls_hs" in name.lower() or "hls_hs" in overlay.ip_dict[name].get("type", "").lower():
            return name
    raise RuntimeError(f"Cannot find hls_HS IP in overlay.ip_dict: {list(overlay.ip_dict.keys())}")


def _write_u32(ip, offset: int, value: int):
    ip.write(int(offset), int(value) & 0xFFFFFFFF)


def _read_u32(ip, offset: int) -> int:
    return int(ip.read(int(offset))) & 0xFFFFFFFF


def _write_addr64(ip, lo_offset: int, hi_offset: int, addr: int):
    a = int(addr) & 0xFFFFFFFFFFFFFFFF
    _write_u32(ip, lo_offset, a & 0xFFFFFFFF)
    _write_u32(ip, hi_offset, (a >> 32) & 0xFFFFFFFF)


def _env_float(name: str, default: float) -> float:
    s = os.getenv(name, "").strip()
    if not s:
        return float(default)
    try:
        return float(s)
    except Exception:
        return float(default)


def _read_grayscale_u8(path: str) -> np.ndarray:
    try:
        from PIL import Image

        img = Image.open(path).convert("L")
        arr = np.asarray(img, dtype=np.uint8)
        if arr.ndim != 2:
            raise ValueError(f"Expected 2D grayscale image, got shape={arr.shape}")
        return arr
    except Exception:
        pass

    try:
        import cv2  # type: ignore

        arr = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if arr is None:
            raise FileNotFoundError(path)
        if arr.ndim != 2:
            raise ValueError(f"Expected 2D grayscale image, got shape={arr.shape}")
        return arr.astype(np.uint8, copy=False)
    except Exception as e:
        raise RuntimeError(
            f"Failed to read PNG as grayscale using PIL or cv2: {path} ({e})"
        )


def _golden_hs(im1_u8: np.ndarray, im2_u8: np.ndarray, iterations: int, alpha: float):
    im1 = im1_u8.astype(np.float32)
    im2 = im2_u8.astype(np.float32)

    im1p = np.pad(im1, ((1, 1), (1, 1)), mode="edge")

    val_m1_m1 = im1p[:-2, :-2]
    val_m1_0 = im1p[:-2, 1:-1]
    val_m1_p1 = im1p[:-2, 2:]
    val_0_m1 = im1p[1:-1, :-2]
    val_0_p1 = im1p[1:-1, 2:]
    val_p1_m1 = im1p[2:, :-2]
    val_p1_0 = im1p[2:, 1:-1]
    val_p1_p1 = im1p[2:, 2:]

    ix = (-val_m1_m1 + val_m1_p1 - 2 * val_0_m1 + 2 * val_0_p1 - val_p1_m1 + val_p1_p1).astype(
        np.float32
    )
    iy = (-val_m1_m1 - 2 * val_m1_0 - val_m1_p1 + val_p1_m1 + 2 * val_p1_0 + val_p1_p1).astype(
        np.float32
    )
    it = (im2 - im1).astype(np.float32)

    u = np.zeros_like(im1, dtype=np.float32)
    v = np.zeros_like(im1, dtype=np.float32)
    alpha2 = np.float32(alpha * alpha)

    for _ in range(int(iterations)):
        up = np.pad(u, ((1, 1), (1, 1)), mode="edge")
        vp = np.pad(v, ((1, 1), (1, 1)), mode="edge")

        u_avg = (up[:-2, 1:-1] + up[2:, 1:-1] + up[1:-1, :-2] + up[1:-1, 2:]) * np.float32(
            0.25
        )
        v_avg = (vp[:-2, 1:-1] + vp[2:, 1:-1] + vp[1:-1, :-2] + vp[1:-1, 2:]) * np.float32(
            0.25
        )

        den = alpha2 + ix * ix + iy * iy
        den = np.maximum(den, np.float32(1e-6))
        num = ix * u_avg + iy * v_avg + it
        frac = num / den
        u = u_avg - ix * frac
        v = v_avg - iy * frac

    return u, v


def _save_flow_arrows(img_u8: np.ndarray, u: np.ndarray, v: np.ndarray, out_path: Path):
    try:
        import cv2  # type: ignore

        h, w = img_u8.shape
        canvas = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)
        for y in range(0, h, FLOW_ARROWS_STEP):
            for x in range(0, w, FLOW_ARROWS_STEP):
                fx = float(u[y, x])
                fy = float(v[y, x])
                if abs(fx) < 0.1 and abs(fy) < 0.1:
                    continue
                pt1 = (int(x), int(y))
                pt2 = (int(round(x + fx * FLOW_ARROWS_SCALE)), int(round(y + fy * FLOW_ARROWS_SCALE)))
                cv2.arrowedLine(
                    canvas,
                    pt1,
                    pt2,
                    (0, 0, 255),
                    thickness=FLOW_ARROWS_THICKNESS,
                    line_type=8,
                    shift=0,
                    tipLength=FLOW_ARROWS_TIP_LENGTH,
                )
        cv2.imwrite(str(out_path), canvas)
        return
    except Exception:
        pass

    try:
        from PIL import Image, ImageDraw

        h, w = img_u8.shape
        canvas = np.stack([img_u8, img_u8, img_u8], axis=-1).astype(np.uint8, copy=False)
        im = Image.fromarray(canvas, mode="RGB")
        draw = ImageDraw.Draw(im)

        for y in range(0, h, FLOW_ARROWS_STEP):
            for x in range(0, w, FLOW_ARROWS_STEP):
                fx = float(u[y, x])
                fy = float(v[y, x])
                if abs(fx) < 0.1 and abs(fy) < 0.1:
                    continue
                x2 = x + fx * FLOW_ARROWS_SCALE
                y2 = y + fy * FLOW_ARROWS_SCALE
                draw.line([(x, y), (x2, y2)], fill=(255, 0, 0), width=FLOW_ARROWS_THICKNESS)
                dx = x2 - x
                dy = y2 - y
                norm = (dx * dx + dy * dy) ** 0.5
                if norm > 1e-6:
                    ux = dx / norm
                    uy = dy / norm
                    px = -uy
                    py = ux
                    tip = (x2, y2)
                    head_len = max(1.0, FLOW_ARROWS_TIP_LENGTH * norm)
                    base = (x2 - ux * head_len, y2 - uy * head_len)
                    head_w = max(1.0, 0.5 * head_len)
                    left = (base[0] + px * head_w, base[1] + py * head_w)
                    right = (base[0] - px * head_w, base[1] - py * head_w)
                    draw.polygon([tip, left, right], fill=(255, 0, 0))

        im.save(str(out_path))
        return
    except Exception:
        pass

    raise RuntimeError(f"Failed to write flow arrows PNG: {out_path}")


def main():
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
    else:
        script_dir = Path.cwd().resolve()
    bit_path = script_dir / "design_1.bit"
    hwh_path = script_dir / "design_1.hwh"

    if not bit_path.exists():
        raise FileNotFoundError(f"Missing bitstream: {bit_path}")
    if not hwh_path.exists():
        raise FileNotFoundError(f"Missing hwh: {hwh_path}")

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--img1",
        default=str(script_dir / "lkof_im0_256x128.png"),
    )
    parser.add_argument(
        "--img2",
        default=str(script_dir / "lkof_im1_256x128.png"),
    )
    parser.add_argument("--avg_epe_thresh", type=float, default=None)
    parser.add_argument("--max_epe_thresh", type=float, default=None)
    parser.add_argument("--avg_aae_deg_thresh", type=float, default=None)
    args, _unknown = parser.parse_known_args()

    avg_epe_thresh = (
        float(args.avg_epe_thresh)
        if args.avg_epe_thresh is not None
        else _env_float("HS_TB_AVG_EPE_THRESH", 1.0)
    )
    max_epe_thresh = (
        float(args.max_epe_thresh)
        if args.max_epe_thresh is not None
        else _env_float("HS_TB_MAX_EPE_THRESH", 1.0)
    )
    avg_aae_deg_thresh = (
        float(args.avg_aae_deg_thresh)
        if args.avg_aae_deg_thresh is not None
        else _env_float("HS_TB_AVG_AAE_DEG_THRESH", 1.0)
    )

    img1_u8 = _read_grayscale_u8(args.img1)
    img2_u8 = _read_grayscale_u8(args.img2)
    if img1_u8.shape != img2_u8.shape:
        raise ValueError(f"Input image shapes differ: {img1_u8.shape} vs {img2_u8.shape}")

    height, width = img1_u8.shape
    if height != MAX_HEIGHT or width != MAX_WIDTH:
        raise ValueError(f"Image size mismatch: got {width}x{height}, expected {MAX_WIDTH}x{MAX_HEIGHT}")

    ol = Overlay(str(bit_path), download=True)
    ip_name = _find_hls_ip(ol)
    ip = getattr(ol, ip_name)

    n = height * width

    inp1 = allocate(shape=(n,), dtype=np.uint16)
    inp2 = allocate(shape=(n,), dtype=np.uint16)
    vx = allocate(shape=(n,), dtype=np.int16)
    vy = allocate(shape=(n,), dtype=np.int16)

    inp1[:] = img1_u8.reshape(-1).astype(np.uint16)
    inp2[:] = img2_u8.reshape(-1).astype(np.uint16)
    vx[:] = 0
    vy[:] = 0

    inp1.flush()
    inp2.flush()
    vx.flush()
    vy.flush()

    _write_addr64(ip, REG_INP1_IMG_LO, REG_INP1_IMG_HI, int(inp1.physical_address))
    _write_addr64(ip, REG_INP2_IMG_LO, REG_INP2_IMG_HI, int(inp2.physical_address))
    _write_addr64(ip, REG_VX_IMG_LO, REG_VX_IMG_HI, int(vx.physical_address))
    _write_addr64(ip, REG_VY_IMG_LO, REG_VY_IMG_HI, int(vy.physical_address))
    _write_u32(ip, REG_HEIGHT, int(height))
    _write_u32(ip, REG_WIDTH, int(width))

    _write_u32(ip, REG_CTRL, 1)
    t0 = time.time()
    while True:
        ctrl = _read_u32(ip, REG_CTRL)
        if (ctrl >> 1) & 0x1:
            break
        if (time.time() - t0) > 5.0:
            raise TimeoutError("IP did not finish within 5 seconds")
        time.sleep(0.001)

    ret_val = _read_u32(ip, REG_AP_RETURN)

    vx.invalidate()
    vy.invalidate()

    nz_vx = int(np.count_nonzero(vx))
    nz_vy = int(np.count_nonzero(vy))
    mean_abs_vx = float(np.mean(np.abs(vx.astype(np.int32))))
    mean_abs_vy = float(np.mean(np.abs(vy.astype(np.int32))))

    vx_f = vx.astype(np.float32) / np.float32(1 << FLOW_FRAC_BITS)
    vy_f = vy.astype(np.float32) / np.float32(1 << FLOW_FRAC_BITS)

    out_flow_path = script_dir / OUT_FLOW_ARROWS_NAME
    _save_flow_arrows(
        img1_u8,
        vx_f.reshape(height, width),
        vy_f.reshape(height, width),
        out_flow_path,
    )

    gold_u, gold_v = _golden_hs(img1_u8, img2_u8, iterations=N_ITER, alpha=ALPHA_VAL)
    gold_u = gold_u.reshape(-1)
    gold_v = gold_v.reshape(-1)

    err_u = np.abs(vx_f - gold_u)
    err_v = np.abs(vy_f - gold_v)
    max_err_u = float(err_u.max())
    max_err_v = float(err_v.max())
    mean_err_u = float(err_u.mean())
    mean_err_v = float(err_v.mean())

    epe = np.sqrt((vx_f - gold_u) ** 2 + (vy_f - gold_v) ** 2)
    avg_epe = float(epe.mean())
    max_epe = float(epe.max())

    dot = 1.0 + vx_f * gold_u + vy_f * gold_v
    mag_hls = np.sqrt(1.0 + vx_f * vx_f + vy_f * vy_f)
    mag_gold = np.sqrt(1.0 + gold_u * gold_u + gold_v * gold_v)
    val = dot / (mag_hls * mag_gold)
    val = np.clip(val, -1.0, 1.0)
    aae_deg = np.arccos(val) * (180.0 / np.pi)
    avg_aae_deg = float(aae_deg.mean())

    print(f"Overlay loaded: {bit_path.name}")
    print(f"IP instance: {ip_name} ({ol.ip_dict[ip_name].get('type', '')})")
    print(f"Images: {args.img1} , {args.img2} ({width}x{height})")
    print(f"Flow arrows: {out_flow_path}")
    print(f"Return = {ret_val}")
    print(f"vx: nonzero={nz_vx}/{n}, mean(|vx|)={mean_abs_vx:.3f}, min={int(vx.min())}, max={int(vx.max())}")
    print(f"vy: nonzero={nz_vy}/{n}, mean(|vy|)={mean_abs_vy:.3f}, min={int(vy.min())}, max={int(vy.max())}")
    print(f"Max Error U: {max_err_u}")
    print(f"Max Error V: {max_err_v}")
    print(f"Mean Error U: {mean_err_u}")
    print(f"Mean Error V: {mean_err_v}")
    print("=============================================")
    print("  HLS Evaluation Metrics (vs Float Golden)")
    print("=============================================")
    print(f"  Average Endpoint Error (AEE): {avg_epe} pixels")
    print(f"  Max Endpoint Error (EPE):     {max_epe} pixels")
    print(f"  Average Angular Error (AAE):  {avg_aae_deg} degrees")
    print("=============================================")
    print(
        "TB Thresholds: "
        f"AVG_EPE<={avg_epe_thresh}, "
        f"MAX_EPE<={max_epe_thresh}, "
        f"AVG_AAE_DEG<={avg_aae_deg_thresh}"
    )

    if ret_val is not None and ret_val != 0:
        raise RuntimeError(f"IP returned non-zero status: {ret_val}")
    if (nz_vx + nz_vy) == 0:
        raise RuntimeError("Output appears all-zero; likely did not run or DDR not connected")
    passed = (avg_epe <= avg_epe_thresh) and (max_epe <= max_epe_thresh) and (avg_aae_deg <= avg_aae_deg_thresh)
    if passed:
        print("Test PASSED!")
    else:
        print("Test FAILED (threshold exceeded).")
        raise RuntimeError(
            f"AVG_EPE={avg_epe} (<= {avg_epe_thresh}), "
            f"MAX_EPE={max_epe} (<= {max_epe_thresh}), "
            f"AVG_AAE_DEG={avg_aae_deg} (<= {avg_aae_deg_thresh})"
        )

    inp1.freebuffer()
    inp2.freebuffer()
    vx.freebuffer()
    vy.freebuffer()


if __name__ == "__main__":
    main()
